In [ ]:
# ============================================
# NOTEBOOK 2: Bone Diagram + MediaPipe Processing
# ============================================
# Yêu cầu: Import dataset từ Notebook 1

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Cài đặt MediaPipe (Kaggle cần cài)
!pip uninstall -y mediapipe
!pip install -q mediapipe==0.10.14

import mediapipe as mp

# Cấu hình
IMG_SIZE = 512  # Kích thước bone diagram
INPUT_DATA_DIR = "/kaggle/input/datasets/tonirighthere/processed-data"  # Dataset từ notebook 1
OUTPUT_DIR = "/kaggle/working/bone_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/bone_images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/keypoints", exist_ok=True)

print("=" * 50)
print("GIAI ĐOẠN 2: BONE DIAGRAM + MEDIAPIPE")
print("=" * 50)

# Khởi tạo MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
# 1. THÊM MÀU SẮC ĐỂ CNN KHÔNG BỊ NHẦM LẪN (M/N, U/V, S/T,..)
FINGER_COLORS = {
    'thumb': (0, 255, 255),    # Vàng
    'index': (255, 0, 255),    # Tím
    'middle': (0, 255, 0),     # Xanh lục
    'ring': (255, 0, 0),       # Xanh lam
    'pinky': (0, 165, 255),    # Cam
    'palm': (255, 255, 255)    # Trắng
}
# Định nghĩa các kết nối xương
HAND_CONNECTIONS = [
    # Cổ tay tới gốc ngón (palm)
    (0, 1, 'palm'), (0, 5, 'palm'), (0, 9, 'palm'), (0, 13, 'palm'), (0, 17, 'palm'),
    (5, 9, 'palm'), (9, 13, 'palm'), (13, 17, 'palm'),
    # Ngón cái
    (1, 2, 'thumb'), (2, 3, 'thumb'), (3, 4, 'thumb'),
    # Ngón trỏ
    (5, 6, 'index'), (6, 7, 'index'), (7, 8, 'index'),
    # Ngón giữa
    (9, 10, 'middle'), (10, 11, 'middle'), (11, 12, 'middle'),
    # Ngón áp út
    (13, 14, 'ring'), (14, 15, 'ring'), (15, 16, 'ring'),
    # Ngón út
    (17, 18, 'pinky'), (18, 19, 'pinky'), (19, 20, 'pinky')
]

def create_bone_diagram(keypoints, img_size=224):
    """Tạo bone diagram từ keypoints với độ phân giải tốt hơn và nét vẽ mỏng hơn"""
    img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
    
    if len(keypoints) == 0:
        return img
    
    h, w = img_size, img_size
    scaled_pts = []
    
    xs = [pt[0] for pt in keypoints]
    ys = [pt[1] for pt in keypoints]
    
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    
    # Scale để bàn tay luôn nằm mượt giữa ảnh
    scale = max(max_x - min_x, max_y - min_y) * 1.5
    if scale == 0: scale = 1.0
    
    center_x = (min_x + max_x) / 2
    center_y = (min_y + max_y) / 2
    
    for pt in keypoints:
        norm_x = (pt[0] - center_x) / scale + 0.5
        norm_y = (pt[1] - center_y) / scale + 0.5
        scaled_pts.append((int(norm_x * w), int(norm_y * h)))
    
    # Vẽ các kết nối (xương)
    for connection in HAND_CONNECTIONS:
        pt1_idx, pt2_idx, part = connection
        if pt1_idx < len(scaled_pts) and pt2_idx < len(scaled_pts):
            pt1 = scaled_pts[pt1_idx]
            pt2 = scaled_pts[pt2_idx]
            color = FINGER_COLORS[part]
            # Sửa thickness=1 để nét mỏng, không bị dính vào nhau
            cv2.line(img, pt1, pt2, color, 1, cv2.LINE_AA)
    
    for pt in scaled_pts:
        cv2.circle(img, pt, 2, (255, 255, 255), -1)
    
    return img

def extract_keypoints_and_bone(image_path):
    """Trích xuất keypoints và tạo bone diagram"""
    img = cv2.imread(image_path)
    if img is None:
        return None, None
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    
    if not results.multi_hand_landmarks:
        return None, None
    
    hand_landmarks = results.multi_hand_landmarks[0]
    keypoints = []
    
    # 2. CHUẨN HÓA TỌA ĐỘ THEO CỔ TAY (WRIST)
    wrist_x = hand_landmarks.landmark[0].x
    wrist_y = hand_landmarks.landmark[0].y
    wrist_z = hand_landmarks.landmark[0].z
    
    for lm in hand_landmarks.landmark:
        # Trừ đi tọa độ gốc của cổ tay để mô hình không bị ảnh hưởng bởi vị trí tay trên camera
        rel_x = lm.x - wrist_x
        rel_y = lm.y - wrist_y
        rel_z = lm.z - wrist_z
        keypoints.append([rel_x, rel_y, rel_z])
    
    bone_img = create_bone_diagram(keypoints, IMG_SIZE)
    
    return bone_img, keypoints

# Đọc metadata
with open(f"{INPUT_DATA_DIR}/metadata.json", 'r') as f:
    metadata = json.load(f)

classes = metadata['classes']
print(f"Classes: {len(classes)}")
print(f"Image size: {metadata['img_size']}")

# Xử lý dữ liệu
print("\n1. Đang tạo bone diagram từ ảnh...")

all_keypoints = []
bone_info = []

for class_name in tqdm(classes, desc="Processing classes"):
    class_dir = f"{INPUT_DATA_DIR}/asl_processed/images/{class_name}"
    if not os.path.exists(class_dir):
        continue
    
    output_class_dir = f"{OUTPUT_DIR}/bone_images/{class_name}"
    os.makedirs(output_class_dir, exist_ok=True)
    
    images = os.listdir(class_dir)
    
    for img_name in images[:500]:  # Giới hạn 500 ảnh/class cho bone diagram
        img_path = f"{class_dir}/{img_name}"
        
        bone_img, keypoints = extract_keypoints_and_bone(img_path)
        
        if bone_img is not None:
            # Lưu bone diagram
            bone_path = f"{output_class_dir}/{img_name}"
            cv2.imwrite(bone_path, bone_img)
            
            # Lưu keypoints
            all_keypoints.append({
                'class': class_name,
                'image': img_name,
                'keypoints': keypoints
            })
            
            bone_info.append({
                'class': class_name,
                'image': img_name,
                'bone_path': bone_path
            })

print(f"\nĐã tạo {len(bone_info)} bone diagrams")

# Lưu keypoints data
keypoints_df = pd.DataFrame(all_keypoints)
keypoints_df.to_csv(f"{OUTPUT_DIR}/keypoints_data.csv", index=False)

# Lưu keypoints dạng numpy (cho BiLSTM)
keypoints_array = []
labels_array = []

class_to_idx = {v: int(k) for k, v in metadata['class_mapping'].items()}  # Đảo ngược mapping

for item in all_keypoints:
    keypoints_array.append(item['keypoints'])
    labels_array.append(class_to_idx[item['class']])

np.save(f"{OUTPUT_DIR}/keypoints_array.npy", np.array(keypoints_array))
np.save(f"{OUTPUT_DIR}/labels_array.npy", np.array(labels_array))

# Lưu metadata cho bone diagram
bone_metadata = {
    'img_size': IMG_SIZE,
    'num_classes': len(classes),
    'total_samples': len(bone_info),
    'hand_connections': HAND_CONNECTIONS,
    'num_keypoints': 42,
    'class_mapping': metadata['class_mapping']
}

with open(f"{OUTPUT_DIR}/bone_metadata.json", 'w') as f:
    json.dump(bone_metadata, f, indent=2)

# 2. Visualize kết quả
print("\n2. Hiển thị bone diagram mẫu:")

fig, axes = plt.subplots(3, 9, figsize=(18, 7))
axes = axes.ravel()

for i, class_name in enumerate(classes[:27]):
    bone_dir = f"{OUTPUT_DIR}/bone_images/{class_name}"
    if os.path.exists(bone_dir):
        samples = os.listdir(bone_dir)
        if samples:
            bone_img = cv2.imread(f"{bone_dir}/{samples[0]}")
            bone_img_rgb = cv2.cvtColor(bone_img, cv2.COLOR_BGR2RGB)
            axes[i].imshow(bone_img_rgb)
            axes[i].set_title(f"{class_name}")
            axes[i].axis('off')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/bone_samples.png")
plt.show()

# 3. So sánh ảnh gốc vs bone diagram
print("\n3. So sánh ảnh gốc và bone diagram:")

fig, axes = plt.subplots(2, 5, figsize=(15, 7))

for i, class_name in enumerate(classes[:5]):
    # Ảnh gốc
    original_dir = f"{INPUT_DATA_DIR}/asl_processed/images/{class_name}"
    if os.path.exists(original_dir):
        original_samples = os.listdir(original_dir)
        if original_samples:
            original_img = cv2.imread(f"{original_dir}/{original_samples[0]}")
            original_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
            axes[0, i].imshow(original_rgb)
            axes[0, i].set_title(f"Original {class_name}")
            axes[0, i].axis('off')
    
    # Bone diagram
    bone_dir = f"{OUTPUT_DIR}/bone_images/{class_name}"
    if os.path.exists(bone_dir):
        bone_samples = os.listdir(bone_dir)
        if bone_samples:
            bone_img = cv2.imread(f"{bone_dir}/{bone_samples[0]}")
            bone_rgb = cv2.cvtColor(bone_img, cv2.COLOR_BGR2RGB)
            axes[1, i].imshow(bone_rgb)
            axes[1, i].set_title(f"Bone {class_name}")
            axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/comparison.png")
plt.show()

# 4. Thống kê
print("\n4. Thống kê dữ liệu bone diagram:")
print(f"Tổng số bone images: {len(bone_info)}")
print(f"Kích thước mỗi ảnh: {IMG_SIZE}x{IMG_SIZE}")
print(f"Số keypoints/mẫu: 42")

# Phân phối dữ liệu theo class
bone_df = pd.DataFrame(bone_info)
distribution = bone_df['class'].value_counts()
print(f"\nPhân phối dữ liệu:")
print(distribution.head(24))

# Plot distribution
plt.figure(figsize=(12, 5))
distribution[:29].plot(kind='bar')
plt.title('Số lượng bone diagram mỗi class')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/bone_distribution.png")
plt.show()

# Lưu thông tin hoàn chỉnh
summary = {
    'stage': 2,
    'total_bone_images': len(bone_info),
    'img_size': IMG_SIZE,
    'keypoints_per_sample': 42,
    'classes_processed': len(classes),
    'output_dir': OUTPUT_DIR
}

with open(f"{OUTPUT_DIR}/stage2_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✅ HOÀN THÀNH GIAI ĐOẠN 2!")
print(f"Bone diagrams saved to: {OUTPUT_DIR}/bone_images")
print(f"Keypoints saved to: {OUTPUT_DIR}/keypoints_data.csv")
print("\n📌 Lưu ý: Save version và add /kaggle/working/bone_data làm dataset output")